In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# LightGBM Stacking Pipeline: 3 Grouped Acuity Classes (ESI 1, ESI 2&3, ESI 4&5) (`models/train_oof_logistic_regression_stacking_exp.ipynb`)

This notebook implements the **Hierarchical LightGBM Stacking Pipeline** with target grouping into **3 Clinical Acuity Categories**:

### 🏥 3-Class Acuity Grouping
1. **Class 1 (`ESI 1`)**: Critical / Resuscitation (Immediate life threat)
2. **Class 2 (`ESI 2&3`)**: Emergent & Urgent (High resource / potentially unstable)
3. **Class 3 (`ESI 4&5`)**: Less Urgent & Non-Urgent (Low resource / stable)

### 🔬 Pipeline Specifications
- **Stratified 3-Way Data Split**: Directly configured from `config/triage_conf.json` (`test_size: 0.1` Holdout Test, `val_size: 0.2` Validation, `Train: 70%`).
- **Strict Complete-Cases Filtering**: Drops any row containing $\ge 1$ null/NA value across all 16 raw features (including `triage_vital_dbp`).
- **Curated 42-Feature Matrix**: 12 core raw measurements + 30 clinically derived engineered features (thresholds, ranges, ratios, interactions).
- **Sub-Model Architecture (2 Layers for 3 Groups)**:
  - **Layer 1 (`L1`)**: `ESI 1` vs `(ESI 2&3 + ESI 4&5)` (Critical Acuity Detector with SMOTE)
  - **Layer 2 (`L2`)**: `ESI 2&3` vs `ESI 4&5` (Urgent vs Non-Urgent Acuity Separator with SMOTE)
- **Optuna Hyperparameter Tuning**: Optimizes sub-model hyperparameters targeting **3-Class Validation Set Macro Balanced Accuracy**.
- **Meta-Learner**: 3-Class Multinomial `LogisticRegression(class_weight='balanced')` calibrated via 5-Fold Cross-Validation on the Validation set.
- **Full Holdout Test Benchmark**: 3x3 Confusion matrix, 42 individual density PNGs, ROC-AUC curves, and Optuna trajectory.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Raw Dataset, Filter Complete Cases & 3-Class Grouping (ESI 1, ESI 2&3, ESI 4&5)
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)

set.seed(config$training$random_state)

stratified_sample <- function(y, fraction, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  sampled <- unlist(lapply(idx_list, function(idx) {
    n_sample <- max(1, round(length(idx) * fraction))
    sample(idx, size = n_sample)
  }))
  return(sort(sampled))
}

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col

initial_total_rows <- nrow(raw_df)
cat("========================================================================\n")
cat(sprintf("  INITIAL DATASET LOADED: %d Total Rows, %d Total Columns\n", initial_total_rows, ncol(raw_df)))
cat("========================================================================\n")

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi_char <- as.character(raw_df[[target_col_name]])

# 3-Class Grouping: 1 -> 1 (ESI 1), 2|3 -> 2 (ESI 2&3), 4|5 -> 3 (ESI 4&5)
grouped_esi_char <- ifelse(is.na(raw_esi_char), NA,
                    ifelse(raw_esi_char == "1", "1",
                    ifelse(raw_esi_char %in% c("2", "3"), "2",
                    ifelse(raw_esi_char %in% c("4", "5"), "3", NA))))

df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  target_col              = factor(grouped_esi_char, levels = c("1", "2", "3"))
)

# Strictly drop any row containing at least 1 null/NA value across all 16 raw features or target
df_master <- na.omit(df_master)
df_master$target_num <- as.numeric(as.character(df_master$target_col))

clean_total_rows <- nrow(df_master)
dropped_rows     <- initial_total_rows - clean_total_rows

cat(sprintf("Missing Values Filter: Dropped %d rows with >= 1 NA feature (Retained %d Complete Cases, %.2f%%)\n", 
            dropped_rows, clean_total_rows, (clean_total_rows / initial_total_rows) * 100))
cat("Grouped 3-Class ESI Distribution (1=ESI 1, 2=ESI 2&3, 3=ESI 4&5):\n")
print(table(df_master$target_col))
cat("------------------------------------------------------------------------\n")

# Stratified 3-Way Partitioning dynamically configured from triage_conf.json
test_size <- config$training$test_size
val_size  <- config$training$val_size
seed_val  <- config$training$random_state

# 1. Extract Stratified Holdout Test Set (e.g. 10%)
idx_test <- stratified_sample(df_master$target_col, test_size, seed = seed_val)
test_df_clean  <- df_master[idx_test, ]
rem_df         <- df_master[-idx_test, ]

# 2. Extract Stratified Validation Set from remainder (e.g. 20% of total)
val_adj_fraction <- val_size / (1 - test_size)
idx_val <- stratified_sample(rem_df$target_col, val_adj_fraction, seed = seed_val + 1)
val_df_clean   <- rem_df[idx_val, ]
train_df_clean <- rem_df[-idx_val, ]

train_mat_export <- as.matrix(cbind(train_df_clean[, 1:16], target = train_df_clean$target_num))
val_mat_export   <- as.matrix(cbind(val_df_clean[, 1:16],   target = val_df_clean$target_num))
test_mat_export  <- as.matrix(cbind(test_df_clean[, 1:16],  target = test_df_clean$target_num))

cat(sprintf("3-Way Partition Complete:\n  Train Set      = %d rows (%.2f%%)\n  Validation Set = %d rows (%.2f%%)\n  Holdout Test   = %d rows (%.2f%%)\n", 
            nrow(train_mat_export), (nrow(train_mat_export) / clean_total_rows) * 100,
            nrow(val_mat_export),   (nrow(val_mat_export) / clean_total_rows) * 100,
            nrow(test_mat_export),  (nrow(test_mat_export) / clean_total_rows) * 100))
cat("========================================================================\n")

In [ ]:
# ---------------------------------------------------------
# Step 2: Build Curated 42-Feature Matrix & Preprocessing
# ---------------------------------------------------------
import os
import pickle
import numpy as np
import pandas as pd
from rpy2.robjects import r
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, balanced_accuracy_score
import lightgbm as lgb
import optuna

train_mat_in = np.array(r('train_mat_export'), dtype=np.float64)
val_mat_in   = np.array(r('val_mat_export'),   dtype=np.float64)
test_mat_in  = np.array(r('test_mat_export'),  dtype=np.float64)

raw_mat_tr  = train_mat_in[:, :16]
y_train     = train_mat_in[:, 16].astype(int) # 1=ESI 1, 2=ESI 2&3, 3=ESI 4&5

raw_mat_val = val_mat_in[:, :16]
y_val       = val_mat_in[:, 16].astype(int)

raw_mat_ts  = test_mat_in[:, :16]
y_test      = test_mat_in[:, 16].astype(int)

def build_42_feature_matrix(raw_mat):
    N = len(raw_mat)
    X = np.zeros((N, 42), dtype=np.float64)
    
    # Extract raw base columns
    age       = raw_mat[:, 0]
    cc_bd     = raw_mat[:, 1]
    gender    = raw_mat[:, 2]
    t_hr      = raw_mat[:, 3]
    t_sbp     = raw_mat[:, 4]
    t_dbp     = raw_mat[:, 5]
    t_rr      = raw_mat[:, 6]
    t_o2      = raw_mat[:, 7]
    pulse_min = raw_mat[:, 8]
    resp_min  = raw_mat[:, 9]
    spo2_min  = raw_mat[:, 10]
    sbp_min   = raw_mat[:, 11]
    pulse_max = raw_mat[:, 12]
    resp_max  = raw_mat[:, 13]
    spo2_max  = raw_mat[:, 14]
    sbp_max   = raw_mat[:, 15]
    
    hr_rng   = pulse_max - pulse_min
    rr_rng   = resp_max - resp_min
    spo2_rng = spo2_max - spo2_min
    sbp_rng  = sbp_max - sbp_min
    
    # 1..12: Specified Raw Features (including triage_vital_dbp)
    X[:, 0]  = age
    X[:, 1]  = cc_bd
    X[:, 2]  = gender
    X[:, 3]  = t_hr
    X[:, 4]  = t_sbp
    X[:, 5]  = t_dbp
    X[:, 6]  = t_rr
    X[:, 7]  = pulse_min
    X[:, 8]  = resp_min
    X[:, 9]  = spo2_min
    X[:, 10] = pulse_max
    X[:, 11] = spo2_max
    
    # 13..22: Baseline Threshold Flags
    is_dyspnea_tot  = (t_o2 < 90).astype(float)
    is_dyspnea_mod  = ((t_o2 >= 90) & (t_o2 < 94)).astype(float)
    is_brady_pnea   = (t_rr < 10).astype(float)
    is_tachy_pnea   = (t_rr > 30).astype(float)
    is_hypo_tension = (t_sbp <= 90).astype(float)
    is_hyper_tension= (t_sbp > 220).astype(float)
    is_brady_tot    = (t_hr < 40).astype(float)
    is_brady_mod    = ((t_hr >= 40) & (t_hr < 60)).astype(float)
    is_tachy_tot    = (t_hr > 150).astype(float)
    is_tachy_mod    = ((t_hr >= 100) & (t_hr <= 150)).astype(float)
    
    X[:, 12] = is_dyspnea_tot
    X[:, 13] = is_dyspnea_mod
    X[:, 14] = is_brady_pnea
    X[:, 15] = is_tachy_pnea
    X[:, 16] = is_hypo_tension
    X[:, 17] = is_hyper_tension
    X[:, 18] = is_brady_tot
    X[:, 19] = is_brady_mod
    X[:, 20] = is_tachy_tot
    X[:, 21] = is_tachy_mod
    
    # 23..35: Ranges, Mid-to-Triage, Ratios
    X[:, 22] = hr_rng
    X[:, 23] = rr_rng
    X[:, 24] = spo2_rng
    X[:, 25] = sbp_rng
    shock_idx = t_hr / np.where(t_sbp == 0, 1.0, t_sbp)
    X[:, 26] = shock_idx
    X[:, 27] = t_hr - hr_rng
    X[:, 28] = t_sbp - sbp_rng
    X[:, 29] = t_rr - rr_rng
    X[:, 30] = t_o2 - spo2_rng
    X[:, 31] = t_o2 / np.where(t_rr == 0, 1.0, t_rr) # rox_index
    X[:, 32] = spo2_rng / np.where(spo2_max == 0, 1.0, spo2_max) # spo2_drop_ratio
    X[:, 33] = hr_rng / (t_hr + 1.0) # hr_instability_ratio
    X[:, 34] = (t_rr / np.where(t_o2 == 0, 1.0, t_o2)) * 100.0 # bif
    
    # 36..42: Curated Advanced Features
    X[:, 35] = (is_dyspnea_tot + is_dyspnea_mod + is_brady_pnea + is_tachy_pnea +
                is_hypo_tension + is_hyper_tension + is_brady_tot + is_brady_mod +
                is_tachy_tot + is_tachy_mod) # n_abnormal_vitals
    X[:, 36] = (t_hr / 80.0) - (t_sbp / 120.0) # perfusion_gap
    X[:, 37] = np.clip(spo2_min - 90.0, -20.0, 20.0) # resp_reserve
    X[:, 38] = cc_bd * (100.0 - t_o2) # bd_x_o2_deficit
    X[:, 39] = cc_bd * is_tachy_pnea   # bd_x_tachypnea
    X[:, 40] = age * (100.0 - t_o2)   # age_o2_interaction
    X[:, 41] = ((t_hr - 80.0) / 80.0) ** 2 # hr_dev_sq
    
    return X

feature_names_42 = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp', 'triage_vital_rr',
    'pulse_min', 'resp_min', 'spo2_min', 'pulse_max', 'spo2_max',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'hr_range', 'rr_range', 'spo2_range', 'sbp_range',
    'shock_index', 'hr_mid_to_triage', 'sbp_mid_to_triage', 'rr_mid_to_triage', 'spo2_mid_to_triage',
    'rox_index', 'spo2_drop_ratio', 'hr_instability_ratio', 'bif',
    'n_abnormal_vitals', 'perfusion_gap', 'resp_reserve', 'bd_x_o2_deficit', 'bd_x_tachypnea',
    'age_o2_interaction', 'hr_dev_sq'
]

X_train_raw = build_42_feature_matrix(raw_mat_tr)
X_val_raw   = build_42_feature_matrix(raw_mat_val)
X_test_raw  = build_42_feature_matrix(raw_mat_ts)

cont_cols_idx = [0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 37, 38, 40, 41]

scaler = StandardScaler()
X_train = X_train_raw.copy()
X_val   = X_val_raw.copy()
X_test  = X_test_raw.copy()

X_train[:, cont_cols_idx] = scaler.fit_transform(X_train_raw[:, cont_cols_idx])
X_val[:, cont_cols_idx]   = scaler.transform(X_val_raw[:, cont_cols_idx])
X_test[:, cont_cols_idx]  = scaler.transform(X_test_raw[:, cont_cols_idx])

def numpy_smote(X, y_bin, seed=42):
    np.random.seed(seed)
    pos_mask = (y_bin == 1)
    neg_mask = (y_bin == 0)
    n_pos = np.sum(pos_mask)
    n_neg = np.sum(neg_mask)
    if n_pos == 0 or n_neg == 0 or n_pos == n_neg:
        return X, y_bin
    if n_pos < n_neg:
        min_X = X[pos_mask]; target_syn = n_neg - n_pos; min_label = 1
    else:
        min_X = X[neg_mask]; target_syn = n_pos - n_neg; min_label = 0
    n_min = len(min_X)
    syn_X = np.zeros((target_syn, X.shape[1]))
    for i in range(target_syn):
        idx1 = np.random.randint(0, n_min)
        idx2 = np.random.randint(0, n_min)
        alpha = np.random.rand()
        syn_X[i] = min_X[idx1] + alpha * (min_X[idx2] - min_X[idx1])
    return np.vstack([X, syn_X]), np.hstack([y_bin, np.full(target_syn, min_label)])

def compute_macro_balanced_accuracy_3class(y_true, y_pred):
    classes = [1, 2, 3]
    bal_accs = []
    for cls in classes:
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal_accs.append((rec + spec) / 2.0)
    return np.mean(bal_accs)

print(f"Feature Matrix Extracted: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
# ---------------------------------------------------------
# Step 2.5: Optuna Hyperparameter Tuning for 3-Class Hierarchical Sub-Models
# ---------------------------------------------------------
# Pre-generate SMOTE training sets for fast trial iterations
# Layer 1: ESI 1 vs (ESI 2..5)
X_sm1, y_sm1 = numpy_smote(X_train, (y_train == 1).astype(int))

# Layer 2: ESI 2&3 (Class 2) vs ESI 4&5 (Class 3) trained on non-ESI 1 data
m2_tr  = (y_train != 1); m2_val = (y_val != 1)
X_sm2, y_sm2 = numpy_smote(X_train[m2_tr], (y_train[m2_tr] == 2).astype(int))

def lgbm_tuning_objective_3class(trial):
    def get_sub_params(prefix):
        return {
            'objective': 'binary',
            'metric': 'binary_logloss',
            'learning_rate': trial.suggest_float(f'{prefix}_lr', 0.02, 0.15, log=True),
            'num_leaves': trial.suggest_int(f'{prefix}_num_leaves', 15, 63),
            'max_depth': trial.suggest_int(f'{prefix}_max_depth', 3, 8),
            'min_child_samples': trial.suggest_int(f'{prefix}_min_child_samples', 10, 80),
            'feature_fraction': trial.suggest_float(f'{prefix}_feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float(f'{prefix}_bagging_fraction', 0.6, 1.0),
            'bagging_freq': 1,
            'reg_alpha': trial.suggest_float(f'{prefix}_reg_alpha', 1e-6, 5.0, log=True),
            'reg_lambda': trial.suggest_float(f'{prefix}_reg_lambda', 1e-6, 5.0, log=True),
            'verbosity': -1,
            'random_state': 42
        }
        
    p_l1 = get_sub_params('l1')
    p_l2 = get_sub_params('l2')
    
    # Fit Sub-Model 1 (ESI 1 vs non-ESI 1)
    m1 = lgb.LGBMClassifier(**p_l1, n_estimators=80)
    m1.fit(X_sm1, y_sm1, eval_set=[(X_val, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(8, verbose=False)])
    p1_v = m1.predict_proba(X_val)[:, 1]
    
    # Fit Sub-Model 2 (ESI 2&3 vs ESI 4&5)
    m2 = lgb.LGBMClassifier(**p_l2, n_estimators=80)
    m2.fit(X_sm2, y_sm2, eval_set=[(X_val[m2_val], (y_val[m2_val] == 2).astype(int))], callbacks=[lgb.early_stopping(8, verbose=False)])
    p2_v = m2.predict_proba(X_val)[:, 1]
    
    # Construct Probability Matrix for 3 Grouped Classes on Validation Set
    val_probs_t = np.zeros((len(X_val), 3))
    val_probs_t[:, 0] = p1_v                  # P(ESI 1)
    val_probs_t[:, 1] = (1 - p1_v) * p2_v      # P(ESI 2&3)
    val_probs_t[:, 2] = (1 - p1_v) * (1 - p2_v)# P(ESI 4&5)
    
    # 5-Fold Stratified Cross-Validation on Validation Set Meta-Learner
    skf_opt = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds_val = np.zeros(len(val_probs_t))
    
    for v_tr, v_te in skf_opt.split(val_probs_t, y_val):
        lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
        lr.fit(val_probs_t[v_tr], y_val[v_tr])
        oof_preds_val[v_te] = lr.predict(val_probs_t[v_te])
        
    # Return 3-Class Validation Set Macro Balanced Accuracy
    return compute_macro_balanced_accuracy_3class(y_val, oof_preds_val)

N_TRIALS = 35
print(f"Starting Optuna Sub-Model Hyperparameter Tuning for 3-Class Grouping ({N_TRIALS} Trials)...")
optuna.logging.set_verbosity(optuna.logging.WARNING)
lgbm_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
lgbm_study.optimize(lgbm_tuning_objective_3class, n_trials=N_TRIALS, timeout=None, show_progress_bar=True)

print("========================================================================")
print(f"OPTUNA 3-CLASS HYPERPARAMETER TUNING COMPLETED!")
print(f"Best Trial #{lgbm_study.best_trial.number}: Peak Validation Macro Balanced Accuracy = {lgbm_study.best_value:.4f}")
print("========================================================================")

In [ ]:
# ---------------------------------------------------------
# Step 2.6: Train Production Sub-Models with Optimal Hyperparameters & Meta-Learner
# ---------------------------------------------------------
best_hparams = lgbm_study.best_trial.params

def extract_best_params(prefix):
    return {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'learning_rate': best_hparams[f'{prefix}_lr'],
        'num_leaves': best_hparams[f'{prefix}_num_leaves'],
        'max_depth': best_hparams[f'{prefix}_max_depth'],
        'min_child_samples': best_hparams[f'{prefix}_min_child_samples'],
        'feature_fraction': best_hparams[f'{prefix}_feature_fraction'],
        'bagging_fraction': best_hparams[f'{prefix}_bagging_fraction'],
        'bagging_freq': 1,
        'reg_alpha': best_hparams[f'{prefix}_reg_alpha'],
        'reg_lambda': best_hparams[f'{prefix}_reg_lambda'],
        'verbosity': -1,
        'random_state': 42
    }

best_p_l1 = extract_best_params('l1')
best_p_l2 = extract_best_params('l2')

print("Training Final Production Sub-Models on Train Set with Optuna-Tuned Hyperparameters...")

# Layer 1: ESI 1 Detector
l1_prod = lgb.LGBMClassifier(**best_p_l1, n_estimators=120)
l1_prod.fit(X_sm1, y_sm1, eval_set=[(X_val, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(12, verbose=False)])

# Layer 2: ESI 2&3 vs ESI 4&5 Separator
l2_prod = lgb.LGBMClassifier(**best_p_l2, n_estimators=120)
l2_prod.fit(X_sm2, y_sm2, eval_set=[(X_val[m2_val], (y_val[m2_val] == 2).astype(int))], callbacks=[lgb.early_stopping(12, verbose=False)])

# Validation Set Probability Matrix (3 Columns)
p1_val_f = l1_prod.predict_proba(X_val)[:, 1]
p2_val_f = l2_prod.predict_proba(X_val)[:, 1]

val_probs_final = np.zeros((len(X_val), 3))
val_probs_final[:, 0] = p1_val_f                  # P(ESI 1)
val_probs_final[:, 1] = (1 - p1_val_f) * p2_val_f      # P(ESI 2&3)
val_probs_final[:, 2] = (1 - p1_val_f) * (1 - p2_val_f)# P(ESI 4&5)

# Fit Final Meta-Learner with Balanced Class Weighting on Validation Set
meta_logreg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_logreg.fit(val_probs_final, y_val)

# Holdout Test Predictions
p1_test = l1_prod.predict_proba(X_test)[:, 1]
p2_test = l2_prod.predict_proba(X_test)[:, 1]

test_probs_prod = np.zeros((len(X_test), 3))
test_probs_prod[:, 0] = p1_test
test_probs_prod[:, 1] = (1 - p1_test) * p2_test
test_probs_prod[:, 2] = (1 - p1_test) * (1 - p2_test)

# Export Bundle
deploy_dir = '../deploy' if os.path.exists('../deploy') else 'deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle_data = {
    'l1_prod': l1_prod,
    'l2_prod': l2_prod,
    'meta_logreg': meta_logreg,
    'target_grouping': {'1': 'ESI 1', '2': 'ESI 2&3', '3': 'ESI 4&5'},
    'best_hyperparameters': best_hparams,
    'scaler_means': scaler.mean_,
    'scaler_sds': scaler.scale_,
    'cont_cols_idx': cont_cols_idx,
    'feature_names': feature_names_42
}

with open(os.path.join(deploy_dir, 'py_oof_stacking_bundle_exp.pkl'), 'wb') as f:
    pickle.dump(bundle_data, f)

print(f"3-Class Grouped Model Bundle saved to: {os.path.join(deploy_dir, 'py_oof_stacking_bundle_exp.pkl')}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Holdout Test Set Evaluation (3 Grouped Classes Breakdown)
# ---------------------------------------------------------
preds_meta_logreg = meta_logreg.predict(test_probs_prod)
probs_meta_logreg = meta_logreg.predict_proba(test_probs_prod)

group_names = {1: 'ESI_1', 2: 'ESI_2&3', 3: 'ESI_4&5'}

def get_per_class_breakdown_3class(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3]
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        try: auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception: auc = 0.0
        recalls.append(rec); specs.append(spec); bal_accs.append(bal); aucs.append(auc)
        rows.append({
            'Pipeline': pipeline_name,
            'Class': group_names[cls],
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)

report_df = get_per_class_breakdown_3class(y_test, preds_meta_logreg, probs_meta_logreg, 'Optuna_Tuned_LightGBM_Stacking_3Groups')

print("========================================================================================")
print("   HOLDOUT TEST REPORT: GROUPED 3-CLASS STACKING (ESI 1 vs ESI 2&3 vs ESI 4&5)")
print("========================================================================================")
print(report_df.to_string(index=False))
print("========================================================================================\n")

reports_dir = '../reports' if os.path.exists('../reports') else 'reports'
os.makedirs(reports_dir, exist_ok=True)

report_df.to_csv(os.path.join(reports_dir, 'oof_multinomial_logistic_stacking_report_exp.csv'), index=False)
print(f"Report saved to {os.path.join(reports_dir, 'oof_multinomial_logistic_stacking_report_exp.csv')}")

In [ ]:
# ---------------------------------------------------------
# Step 4: 3x3 Confusion Matrix Graph for Grouped Acuity Classes
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels_3 = ['ESI 1', 'ESI 2&3', 'ESI 4&5']

# Compute confusion matrix
cm_meta      = confusion_matrix(y_test, preds_meta_logreg, labels=[1, 2, 3])
cm_meta_norm = cm_meta.astype('float') / cm_meta.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(8, 6.5))

annot_meta = np.empty_like(cm_meta, dtype=object)
for i in range(3):
    for j in range(3):
        annot_meta[i, j] = f"{cm_meta[i, j]}\n({cm_meta_norm[i, j]*100:.1f}%)"

sns.heatmap(cm_meta_norm, annot=annot_meta, fmt='', cmap='Greens', cbar=True,
            xticklabels=esi_labels_3, yticklabels=esi_labels_3, ax=ax, vmin=0, vmax=1)
ax.set_title('Holdout Test Confusion Matrix\nGrouped 3-Class Stacking (ESI 1, ESI 2&3, ESI 4&5)', fontsize=12.5, fontweight='bold', pad=12)
ax.set_xlabel('Predicted ESI Group', fontsize=11, fontweight='bold')
ax.set_ylabel('True ESI Group', fontsize=11, fontweight='bold')

plt.tight_layout()

cm_path = os.path.join(plots_dir, 'holdout_test_confusion_matrix_exp.png')
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'holdout_test_confusion_matrix_exp.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"3x3 Confusion Matrix Graph saved to: {cm_path}")

In [ ]:
# ---------------------------------------------------------
# Step 5: Density Data Distribution Graphs (3 Grouped Classes)
# ---------------------------------------------------------
import os
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
density_dir = os.path.join(plots_dir, 'density_exp')
density_img_dir = os.path.join(plots_dir, 'image', 'density_exp')
os.makedirs(density_dir, exist_ok=True)
os.makedirs(density_img_dir, exist_ok=True)

# Construct full DataFrame of 42 features and 3-class ESI labels
df_raw_all = pd.DataFrame(X_train_raw, columns=feature_names_42)
group_map = {1: 'ESI 1', 2: 'ESI 2&3', 3: 'ESI 4&5'}
df_raw_all['ESI'] = [group_map[k] for k in y_train]

esi_palette = {
    'ESI 1': '#d62728',   # Red (Critical / Resuscitation)
    'ESI 2&3': '#ff7f0e', # Orange (Emergent / Urgent)
    'ESI 4&5': '#2ca02c'  # Green (Less Urgent / Non-Urgent)
}

print(f"Saving individual feature density distribution plots ({len(feature_names_42)} features) to: {density_dir}")

for feat in feature_names_42:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    if feat in ['gender', 'cc_breathingdifficulty'] or feat.startswith('is_') or feat.startswith('bd_x_tachypnea'):
        prop_df = df_raw_all.groupby('ESI')[feat].mean().reset_index(name='Proportion')
        sns.barplot(data=prop_df, x='ESI', y='Proportion', palette=esi_palette, ax=ax, edgecolor='black',
                    order=['ESI 1', 'ESI 2&3', 'ESI 4&5'])
        ax.set_title(f"{feat} (Prevalence by 3-Class ESI Group)", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel("ESI Group", fontsize=11, fontweight='bold')
        ax.set_ylabel("Prevalence / Proportion", fontsize=11, fontweight='bold')
        for p in ax.patches:
            ax.annotate(f"{p.get_height()*100:.1f}%",
                        (p.get_x() + p.get_width() / 2., p.get_height()),
                        ha='center', va='bottom', fontsize=10, fontweight='bold', xytext=(0, 2),
                        textcoords='offset points')
    else:
        sns.kdeplot(
            data=df_raw_all,
            x=feat,
            hue='ESI',
            hue_order=['ESI 1', 'ESI 2&3', 'ESI 4&5'],
            palette=esi_palette,
            common_norm=False,
            fill=True,
            alpha=0.20,
            linewidth=2.0,
            ax=ax
        )
        ax.set_title(f"Feature Density Distribution: {feat}", fontsize=13, fontweight='bold', pad=12)
        ax.set_xlabel(feat, fontsize=11, fontweight='bold')
        ax.set_ylabel("Density", fontsize=11, fontweight='bold')
    
    ax.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    
    out_file = f"density_{feat}.png"
    plt.savefig(os.path.join(density_dir, out_file), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(density_img_dir, out_file), dpi=300, bbox_inches='tight')
    plt.close()

print(f"All {len(feature_names_42)} individual density plots successfully saved in {density_dir}")

In [ ]:
# ---------------------------------------------------------
# Step 6: 3-Class Multiclass ROC-AUC Curve Analysis (Holdout Test Benchmark)
# ---------------------------------------------------------
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

plots_dir = '../plots' if os.path.exists('../plots') else 'plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

classes = [1, 2, 3]
class_names_roc = ['ESI 1', 'ESI 2&3', 'ESI 4&5']
y_test_bin = label_binarize(y_test, classes=classes)
n_classes  = len(classes)

fpr = dict()
tpr = dict()
roc_auc = dict()

for i, cls in enumerate(classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs_meta_logreg[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs_meta_logreg.ravel())
roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])

all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
mean_tpr /= n_classes

fpr["macro"] = all_fpr
tpr["macro"] = mean_tpr
roc_auc["macro"] = auc(fpr["macro"], tpr["macro"])

plt.figure(figsize=(9, 8))

esi_colors = {
    0: '#d62728',  # ESI 1: Red
    1: '#ff7f0e',  # ESI 2&3: Orange
    2: '#2ca02c'   # ESI 4&5: Green
}

plt.plot(fpr["micro"], tpr["micro"],
         label=f"Micro-Average (AUC = {roc_auc['micro']:.4f})",
         color='#e377c2', linestyle=':', linewidth=2.5)
plt.plot(fpr["macro"], tpr["macro"],
         label=f"Macro-Average (AUC = {roc_auc['macro']:.4f})",
         color='#17becf', linestyle='--', linewidth=2.5)

for i, cls in enumerate(classes):
    plt.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0,
             label=f"{class_names_roc[i]} (AUC = {roc_auc[i]:.4f})")

plt.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=12, fontweight='bold')
plt.title('Holdout Test ROC-AUC Curves (3-Class Grouping)', fontsize=14, fontweight='bold', pad=12)
plt.legend(loc="lower right", fontsize=10.5, frameon=True, framealpha=0.95)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

roc_plot_path = os.path.join(plots_dir, 'holdout_test_roc_auc_curve_exp.png')
plt.savefig(roc_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'holdout_test_roc_auc_curve_exp.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"ROC-AUC Curve Graph saved to: {roc_plot_path}")

In [ ]:
# ---------------------------------------------------------
# Step 7: Optuna Sub-Model Hyperparameter Tuning Trajectory Graph (3 Groups)
# ---------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

trial_nums = [t.number for t in lgbm_study.trials if t.value is not None]
trial_vals = [t.value for t in lgbm_study.trials if t.value is not None]
running_max = np.maximum.accumulate(trial_vals)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. Optimization Trajectory
ax1.scatter(trial_nums, trial_vals, color='#1f77b4', alpha=0.6, label='Trial Macro Balanced Acc')
ax1.plot(trial_nums, running_max, color='#d62728', linewidth=2.2, label='Best Macro Balanced Acc')
ax1.set_title('Optuna 3-Class Sub-Model Search History', fontsize=12.5, fontweight='bold', pad=12)
ax1.set_xlabel('Trial Number', fontsize=11, fontweight='bold')
ax1.set_ylabel('Val Macro Balanced Accuracy', fontsize=11, fontweight='bold')
ax1.legend(loc='lower right')
ax1.grid(True, linestyle='--', alpha=0.4)

# 2. Summary of Tuned Learning Rates across Sub-Models
sub_model_names = ['L1 (ESI 1 vs 2..5)', 'L2 (ESI 2&3 vs 4&5)']
best_lrs = [
    best_hparams['l1_lr'],
    best_hparams['l2_lr']
]
palette = ['#d62728', '#ff7f0e']

sns.barplot(x=sub_model_names, y=best_lrs, palette=palette, ax=ax2, edgecolor='black')
ax2.set_title('Optuna Optimal Learning Rate per Sub-Model Layer', fontsize=12.5, fontweight='bold', pad=12)
ax2.set_xlabel('LightGBM Sub-Model Layer', fontsize=11, fontweight='bold')
ax2.set_ylabel('Learning Rate', fontsize=11, fontweight='bold')
for p in ax2.patches:
    ax2.annotate(f"{p.get_height():.4f}",
                 (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='bottom', fontsize=10.5, fontweight='bold', xytext=(0, 2),
                 textcoords='offset points')
ax2.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
traj_plot_path = os.path.join(plots_dir, 'optuna_lgbm_hyperparameter_trajectory.png')
plt.savefig(traj_plot_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'optuna_lgbm_hyperparameter_trajectory.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Hyperparameter Tuning Trajectory Graph saved to: {traj_plot_path}")